### Imports

In [65]:
# Import the required libraries.

import pandas as pd
import numpy as np
import pickle
import re
import holidays

from sklearn.preprocessing import PowerTransformer
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

### Load the new raw data

In [26]:
# Load the new raw dataset.

df = pd.read_csv("given_data.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (5000, 35)


,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,...,f26,f27,f28,f29,f30,f31,f32,f33,f34,y
0,Source2,2021-10-24 15:12:44,-73.833107,NaN,NaN,0.000,Right lane blocked due to accident on Hutchins...,Hutchinson River Pkwy S,Bronx,Bronx,...,False,False,False,False,False,False,Day,Day,Day,2
1,Source1,2022-03-11 17:17:00.000000000,-81.925184,26.674854,-81.925661,0.033,Incident on NE PINE ISLAND RD near NE 23RD AVE...,NE Pine Island Rd,Cape Coral,Lee,...,False,False,False,False,False,False,Day,Day,Day,2
2,Source1,2019-11-07 19:42:00,-118.286030,33.916421,-118.286030,0.000,At I-110/Harbor Fwy - Accident.,I-110 S,Gardena,Los Angeles,...,False,False,False,False,True,False,Night,Night,Night,2
3,Source2,2017-11-08 13:52:32,-93.332603,NaN,NaN,0.000,Accident on I-94 I-694 Westbound at Exit 33 I-...,Brooklyn Blvd,Minneapolis,Hennepin,...,False,False,False,False,False,False,Day,Day,Day,3
4,Source1,2018-10-08 11:15:36,-122.135321,43.614425,-122.133587,0.155,At NF-5897 - Accident.,Highway 58,Oakridge,Lane,...,False,False,False,False,False,False,Day,Day,Day,2


## 01_data_understanding :

### Rename the identified columns as 01 file

In [27]:
# Rename the raw columns using the names identified during data understanding.

column_names = {
    "f1": "Source",
    "f2": "Accident_Time",
    "f3": "Additional_Lng",
    "f4": "Lat",
    "f5": "Lng",
    "f7": "Description",
    "f8": "Street",
    "f9": "City",
    "f10": "County",
    "f11": "State",
    "f12": "Zipcode",
    "f13": "Country",
    "f14": "Airport_Code",
    "f15": "Weather_Time",
    "f18": "Wind_Direction"
}

df = df.rename(columns=column_names)

df.head()

,Source,Accident_Time,Additional_Lng,Lat,Lng,f6,Description,Street,City,County,...,f26,f27,f28,f29,f30,f31,f32,f33,f34,y
0,Source2,2021-10-24 15:12:44,-73.833107,NaN,NaN,0.000,Right lane blocked due to accident on Hutchins...,Hutchinson River Pkwy S,Bronx,Bronx,...,False,False,False,False,False,False,Day,Day,Day,2
1,Source1,2022-03-11 17:17:00.000000000,-81.925184,26.674854,-81.925661,0.033,Incident on NE PINE ISLAND RD near NE 23RD AVE...,NE Pine Island Rd,Cape Coral,Lee,...,False,False,False,False,False,False,Day,Day,Day,2
2,Source1,2019-11-07 19:42:00,-118.286030,33.916421,-118.286030,0.000,At I-110/Harbor Fwy - Accident.,I-110 S,Gardena,Los Angeles,...,False,False,False,False,True,False,Night,Night,Night,2
3,Source2,2017-11-08 13:52:32,-93.332603,NaN,NaN,0.000,Accident on I-94 I-694 Westbound at Exit 33 I-...,Brooklyn Blvd,Minneapolis,Hennepin,...,False,False,False,False,False,False,Day,Day,Day,3
4,Source1,2018-10-08 11:15:36,-122.135321,43.614425,-122.133587,0.155,At NF-5897 - Accident.,Highway 58,Oakridge,Lane,...,False,False,False,False,False,False,Day,Day,Day,2


## 02_data_processing :

### Load train-based parameters

In [28]:
# Load preprocessing parameters learned from the training data.

with open("02_processing_params.pkl", "rb") as file:
    processing_params = pickle.load(file)

### Location information

In [29]:
# Create location availability and remove the original coordinates.

def process_location(df):
    df = df.copy()

    df["Lat"] = pd.to_numeric(
        df["Lat"],
        errors="coerce"
    )

    df["Lng"] = pd.to_numeric(
        df["Lng"],
        errors="coerce"
    )

    df["Location_Available"] = (
        df["Lat"].notna()
        & df["Lng"].notna()
    ).astype("int8")

    df.drop(
        columns=["Lat", "Lng"],
        inplace=True
    )

    return df


df = process_location(df)

### Environmental missing values

for f16, f17, f19, f20

In [30]:
# Fill environmental missing values using train-based location statistics.

def apply_hierarchical_imputer(
    df,
    feature,
    imputation_stats,
    city_col="City",
    state_col="State"
):
    city_keys = pd.Series(
        list(zip(df[state_col], df[city_col])),
        index=df.index
    )

    city_estimates = city_keys.map(
        imputation_stats["city_lookup"]
    )

    state_estimates = df[state_col].map(
        imputation_stats["state_lookup"]
    )

    final_estimates = (
        city_estimates
        .fillna(state_estimates)
        .fillna(imputation_stats["global_mean"])
    )

    df[feature] = (
        pd.to_numeric(df[feature], errors="coerce")
        .fillna(final_estimates)
    )


def process_environmental_features(
    df,
    environmental_stats
):
    df = df.copy()

    features = [
        "f16",
        "f17",
        "f19",
        "f20"
    ]

    for feature in features:
        apply_hierarchical_imputer(
            df=df,
            feature=feature,
            imputation_stats=environmental_stats[feature]
        )

    return df


df = process_environmental_features(
    df,
    processing_params["environmental_imputation"]
)

### Normalize Wind Direction

In [31]:
# Normalize different representations of wind direction.

def normalize_wind_direction(value):

    if pd.isna(value):
        return "Unknown"

    value = str(value).strip().upper()

    mapping = {
        "N": "N",
        "NORTH": "N",

        "S": "S",
        "SOUTH": "S",

        "E": "E",
        "EAST": "E",

        "W": "W",
        "WEST": "W",

        "NE": "NE",
        "NORTH EAST": "NE",
        "NORTHEAST": "NE",

        "NW": "NW",
        "NORTH WEST": "NW",
        "NORTHWEST": "NW",

        "SE": "SE",
        "SOUTH EAST": "SE",
        "SOUTHEAST": "SE",

        "SW": "SW",
        "SOUTH WEST": "SW",
        "SOUTHWEST": "SW",

        "ENE": "ENE",
        "EAST NORTHEAST": "ENE",

        "ESE": "ESE",
        "EAST SOUTHEAST": "ESE",

        "NNE": "NNE",
        "NORTH NORTHEAST": "NNE",

        "NNW": "NNW",
        "NORTH NORTHWEST": "NNW",

        "SSE": "SSE",
        "SOUTH SOUTHEAST": "SSE",

        "SSW": "SSW",
        "SOUTH SOUTHWEST": "SSW",

        "WSW": "WSW",
        "WEST SOUTHWEST": "WSW",

        "WNW": "WNW",
        "WEST NORTHWEST": "WNW",

        "VAR": "VARIABLE",
        "VARIABLE": "VARIABLE",
        "VRB": "VARIABLE",

        "CALM": "CALM"
    }

    return mapping.get(value, value)


df["Wind_Direction"] = (
    df["Wind_Direction"]
    .apply(normalize_wind_direction)
)

### Wind Direction Feature Engineering

In [32]:
# Convert wind direction into binary and cyclical features.

def encode_wind_direction(df):
    df = df.copy()

    wind_values = (
        df["Wind_Direction"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    df["Wind_D_Calm"] = (
        wind_values.eq("CALM")
    ).astype("int8")

    df["Wind_D_Variable"] = (
        wind_values.eq("VARIABLE")
    ).astype("int8")

    wind_angle_degrees = {
        "E": 0.0,
        "ENE": 22.5,
        "NE": 45.0,
        "NNE": 67.5,
        "N": 90.0,
        "NNW": 112.5,
        "NW": 135.0,
        "WNW": 157.5,
        "W": 180.0,
        "WSW": 202.5,
        "SW": 225.0,
        "SSW": 247.5,
        "S": 270.0,
        "SSE": 292.5,
        "SE": 315.0,
        "ESE": 337.5
    }

    valid_values = set(wind_angle_degrees) | {
        "CALM",
        "VARIABLE",
        "UNKNOWN"
    }

    unexpected_values = sorted(
        set(wind_values.dropna()) - valid_values
    )

    if unexpected_values:
        raise ValueError(
            f"Unexpected wind directions: {unexpected_values}"
        )

    wind_angles = wind_values.map(
        wind_angle_degrees
    )

    wind_radians = np.deg2rad(
        wind_angles
    )

    df["Wind_D_Sin"] = np.sin(
        wind_radians
    )

    df["Wind_D_Cos"] = np.cos(
        wind_radians
    )

    special_mask = wind_values.isin([
        "CALM",
        "VARIABLE"
    ])

    df.loc[
        special_mask,
        ["Wind_D_Sin", "Wind_D_Cos"]
    ] = 0.0

    unknown_mask = wind_values.eq(
        "UNKNOWN"
    )

    df.loc[
        unknown_mask,
        ["Wind_D_Sin", "Wind_D_Cos"]
    ] = np.nan

    return df


df = encode_wind_direction(df)

### Impute unknown wind directions

In [33]:
# Fill unknown wind components using train-based location statistics.

def impute_wind_features(
    df,
    wind_imputation_stats
):
    df = df.copy()

    for feature in [
        "Wind_D_Sin",
        "Wind_D_Cos"
    ]:
        apply_hierarchical_imputer(
            df=df,
            feature=feature,
            imputation_stats=wind_imputation_stats[feature]
        )

    df[
        ["Wind_D_Sin", "Wind_D_Cos"]
    ] = df[
        ["Wind_D_Sin", "Wind_D_Cos"]
    ].astype("float32")

    return df


df = impute_wind_features(
    df,
    processing_params["wind_imputation"]
)

### Remove original Wind_Direction

In [34]:
# Remove the original wind direction column.

df.drop(
    columns=["Wind_Direction"],
    inplace=True
)

### Create Record_Delay_Time

In [35]:
# Create record delay and fill missing values using the train mean.

def process_record_delay(
    df,
    mean_delay
):
    df = df.copy()

    df["Accident_Time"] = pd.to_datetime(
        df["Accident_Time"],
        format="mixed",
        errors="coerce"
    ).dt.floor("s")

    df["Weather_Time"] = pd.to_datetime(
        df["Weather_Time"],
        errors="coerce"
    )

    df["Record_Delay_Time"] = (
        df["Accident_Time"]
        - df["Weather_Time"]
    ).dt.total_seconds() / 60

    df["Record_Delay_Time"] = (
        df["Record_Delay_Time"]
        .replace(
            [float("inf"), float("-inf")],
            pd.NA
        )
        .fillna(mean_delay)
        .astype("float32")
    )

    df.drop(
        columns=["Weather_Time"],
        inplace=True
    )

    return df


df = process_record_delay(
    df,
    processing_params["mean_delay"]
)

### Remove Zipcode / Airport Code and missing Street rows

In [36]:
# Remove unused location columns and rows without Street information.

def clean_location_columns(df):
    df = df.copy()

    columns_to_drop = [
        "ZipCode",
        "Zipcode",
        "Airport_Code",
        "Airpot_Code"
    ]

    existing_columns = [
        column
        for column in columns_to_drop
        if column in df.columns
    ]

    df.drop(
        columns=existing_columns,
        inplace=True
    )

    df.dropna(
        subset=["Street"],
        inplace=True
    )

    df.reset_index(
        drop=True,
        inplace=True
    )

    return df


df = clean_location_columns(df)

### Rename and encode f32–f34

In [37]:
# Rename and encode day/night features using train-based modes.

def process_day_night_features(
    df,
    day_night_modes
):
    df = df.copy()

    df.rename(
        columns={
            "f32": "D1",
            "f33": "D2",
            "f34": "D3"
        },
        inplace=True
    )

    mapping = {
        "DAY": 1,
        "NIGHT": 0
    }

    for column in [
        "D1",
        "D2",
        "D3"
    ]:
        values = (
            df[column]
            .astype("string")
            .str.strip()
            .str.upper()
            .replace("", pd.NA)
        )

        unexpected_values = sorted(
            set(values.dropna())
            - set(mapping)
        )

        if unexpected_values:
            raise ValueError(
                f"Unexpected values in {column}: "
                f"{unexpected_values}"
            )

        df[column] = (
            values
            .map(mapping)
            .fillna(day_night_modes[column])
            .astype("int8")
        )

    return df


df = process_day_night_features(
    df,
    processing_params["day_night_modes"]
)

### Remove remaining missing rows

In [38]:
# Remove rows that still contain missing feature values.

feature_columns = [
    column
    for column in df.columns
    if column != "y"
]

df = (
    df
    .dropna(subset=feature_columns)
    .reset_index(drop=True)
)

### Remove invalid Additional_Lng values

In [39]:
# Remove rows with invalid longitude values.

def clean_additional_longitude(df):
    df = df.copy()

    df["Additional_Lng"] = pd.to_numeric(
        df["Additional_Lng"],
        errors="coerce"
    )

    valid_mask = (
        df["Additional_Lng"]
        .between(
            -125,
            -66,
            inclusive="both"
        )
    )

    df = (
        df.loc[valid_mask]
        .reset_index(drop=True)
    )

    return df


df = clean_additional_longitude(df)

### IQR Capping

In [40]:
# Limit outliers using IQR boundaries learned from training data.

def apply_iqr_limits(
    df,
    iqr_limits
):
    df = df.copy()

    features = [
        "f6",
        "f16",
        "f17",
        "f19",
        "f20"
    ]

    for feature in features:
        df[feature] = pd.to_numeric(
            df[feature],
            errors="coerce"
        )

        lower = iqr_limits[feature]["lower"]
        upper = iqr_limits[feature]["upper"]

        df[feature] = df[feature].clip(
            lower=lower,
            upper=upper
        )

    return df


df = apply_iqr_limits(
    df,
    processing_params["iqr_limits"]
)

### Convert Source

In [41]:
# Convert Source categories into numerical values.

def encode_source(df):
    df = df.copy()

    source_mapping = {
        "SOURCE1": 1,
        "SOURCE2": 2,
        "SOURCE3": 3
    }

    values = (
        df["Source"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    unexpected_values = sorted(
        set(values.dropna())
        - set(source_mapping)
    )

    if unexpected_values:
        raise ValueError(
            f"Unexpected Source values: "
            f"{unexpected_values}"
        )

    df["Source"] = (
        values
        .map(source_mapping)
        .astype("int8")
    )

    return df


df = encode_source(df)

### Standardization

In [42]:
# Standardize numerical features using train mean and scale values.

def apply_train_standardization(
    df,
    scaler_params
):
    df = df.copy()

    for column, params in scaler_params.items():

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

        df[column] = (
            df[column] - params["mean"]
        ) / params["scale"]

    return df


df = apply_train_standardization(
    df,
    processing_params["scaler"]
)

### Normalize text

In [43]:
# Normalize textual values.

text_columns = [
    "Description",
    "Street",
    "City",
    "County",
    "State",
    "Country"
]

for column in text_columns:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

### Convert Boolean features

In [44]:
# Convert Boolean features into binary values.

def encode_boolean_features(df):
    df = df.copy()

    boolean_columns = [
        f"f{number}"
        for number in range(21, 32)
    ]

    boolean_mapping = {
        "TRUE": 1,
        "FALSE": 0,
        "1": 1,
        "0": 0
    }

    for column in boolean_columns:

        values = (
            df[column]
            .astype("string")
            .str.strip()
            .str.upper()
        )

        unexpected_values = sorted(
            set(values.dropna())
            - set(boolean_mapping)
        )

        if unexpected_values:
            raise ValueError(
                f"Unexpected values in {column}: "
                f"{unexpected_values}"
            )

        df[column] = (
            values
            .map(boolean_mapping)
            .astype("int8")
        )

    return df


df = encode_boolean_features(df)

### Convert float64 → float32

In [45]:
# Reduce floating-point columns to float32.

float_columns = df.select_dtypes(
    include=["float64"]
).columns

df[float_columns] = (
    df[float_columns]
    .astype("float32")
)

## 03_feature_engineering :

### Load feature-engineering parameters

In [47]:
# Load feature engineering parameters learned from training data.

with open(
    "03_feature_engineering_params.pkl",
    "rb"
) as file:

    feature_engineering_params = pickle.load(file)

### Create is_holiday

In [48]:
# Create the holiday/weekend feature from accident time.

def create_holiday_feature(df):
    df = df.copy()

    df["Accident_Time"] = pd.to_datetime(
        df["Accident_Time"],
        errors="coerce"
    )

    years = (
        df["Accident_Time"]
        .dt.year
        .dropna()
        .unique()
    )

    us_holidays = holidays.UnitedStates(
        years=years
    )

    is_public_holiday = (
        df["Accident_Time"]
        .dt.date
        .isin(us_holidays)
    )

    is_weekend = (
        df["Accident_Time"]
        .dt.dayofweek >= 5
    )

    df["is_holiday"] = (
        is_public_holiday |
        is_weekend
    ).astype("int8")

    df.drop(
        columns=["Accident_Time"],
        inplace=True
    )

    return df


df = create_holiday_feature(df)

### Description

In [49]:
description_keywords = [
    "closed",
    "lanes blocked",
    "alternate",
    "alternative",
    "re-opened",
    "express",
    "queueing",
    "tractor",
    "trailer",
    "police"
]

### Create has_sensitive_word

In [50]:
# Detect severity-sensitive words in accident descriptions.

def create_sensitive_word_feature(
    df,
    keywords
):
    df = df.copy()

    keyword_pattern = (
        r"(?<![a-z])(?:"
        + "|".join(
            re.escape(keyword)
            for keyword in keywords
        )
        + r")(?![a-z])"
    )

    normalized_description = (
        df["Description"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.replace(
            r"[‐-‒–—−]",
            "-",
            regex=True
        )
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
    )

    df["has_sensitive_word"] = (
        normalized_description
        .str.contains(
            keyword_pattern,
            regex=True,
            na=False
        )
        .astype("int8")
    )

    df.drop(
        columns=["Description"],
        inplace=True
    )

    return df


description_keywords = [
    "closed",
    "lanes blocked",
    "alternate",
    "alternative",
    "re-opened",
    "express",
    "queueing",
    "tractor",
    "trailer",
    "police"
]


df = create_sensitive_word_feature(
    df,
    description_keywords
)

### Address Features

#### Remove Country

In [51]:
# Remove Country because it contains no useful variation.

df.drop(
    columns=["Country"],
    inplace=True
)

#### Helper for hierarchical address keys

In [52]:
# Create hierarchical address keys.

def make_address_key(
    dataset,
    columns
):
    return pd.Series(
        list(
            map(
                tuple,
                dataset[columns].to_numpy()
            )
        ),
        index=dataset.index
    )

#### Prepare address columns

In [53]:
# Prepare address columns for feature engineering.

def prepare_address_columns(df):
    df = df.copy()

    address_columns = [
        "Street",
        "City",
        "County",
        "State"
    ]

    df[address_columns] = (
        df[address_columns]
        .fillna("__MISSING__")
        .astype(str)
    )

    return df


df = prepare_address_columns(df)

#### Address_Freq

In [54]:
# Create address frequency using frequency maps learned from training data.

def create_address_frequency(
    df,
    frequency_params
):
    df = df.copy()

    street_key = make_address_key(
        df,
        [
            "State",
            "County",
            "City",
            "Street"
        ]
    )

    city_key = make_address_key(
        df,
        [
            "State",
            "County",
            "City"
        ]
    )

    street_frequency = (
        street_key
        .map(
            frequency_params[
                "street_freq_map"
            ]
        )
        .fillna(0)
    )

    city_frequency = (
        city_key
        .map(
            frequency_params[
                "city_freq_map"
            ]
        )
        .fillna(0)
    )

    frequency_scale = (
        frequency_params[
            "frequency_scale"
        ]
    )

    street_score = (
        np.log1p(street_frequency)
        / frequency_scale
    )

    city_score = (
        np.log1p(city_frequency)
        / frequency_scale
    )

    df["Address_Freq"] = (

        frequency_params["street_weight"]
        * street_score

        +

        frequency_params["city_weight"]
        * city_score

    ).astype("float32")

    return df


df = create_address_frequency(
    df,
    feature_engineering_params[
        "address_frequency"
    ]
)

#### Address_Mean

In [55]:
# Create hierarchical target encoding using statistics learned from training data.

def create_address_mean(
    df,
    encoder
):
    df = df.copy()

    global_mean = encoder[
        "global_mean"
    ]

    # State fallback
    state_mean = (
        df["State"]
        .map(
            encoder["state"]
        )
        .fillna(global_mean)
    )

    # County fallback
    county_key = make_address_key(
        df,
        [
            "State",
            "County"
        ]
    )

    county_mean = (
        county_key
        .map(
            encoder["county"]
        )
        .fillna(state_mean)
    )

    # City fallback
    city_key = make_address_key(
        df,
        [
            "State",
            "County",
            "City"
        ]
    )

    city_mean = (
        city_key
        .map(
            encoder["city"]
        )
        .fillna(county_mean)
    )

    # Street fallback
    street_key = make_address_key(
        df,
        [
            "State",
            "County",
            "City",
            "Street"
        ]
    )

    street_mean = (
        street_key
        .map(
            encoder["street"]
        )
        .fillna(city_mean)
    )

    df["Address_Mean"] = (
        street_mean.astype("float32")
    )

    return df


df = create_address_mean(
    df,
    feature_engineering_params[
        "address_target_encoder"
    ]
)

#### Remove original address columns

In [56]:
# Remove the original address columns.

df.drop(
    columns=[
        "Street",
        "City",
        "County",
        "State"
    ],
    inplace=True
)

#### Transform address features

In [57]:
# Apply the address transformation learned from training data.

def transform_address_features(
    df,
    transformer
):
    df = df.copy()

    address_features = [
        "Address_Freq",
        "Address_Mean"
    ]

    df[address_features] = (
        transformer.transform(
            df[address_features]
        )
    )

    return df


df = transform_address_features(
    df,
    feature_engineering_params[
        "address_power_transformer"
    ]
)

### Feature Selection

#### Remove low-information features

In [58]:
# Remove low-information features selected during training.

features_to_drop = [
    "f23",
    "f29",
    "f22",
    "f26",
    "f31"
]

df.drop(
    columns=features_to_drop,
    inplace=True
)

#### Remove Location_Available

In [59]:
# Remove the redundant location availability feature.

df.drop(
    columns=["Location_Available"],
    inplace=True
)

### Final feature check

In [60]:
# Check the final feature set before modeling.

expected_features = [
    "Source",
    "Additional_Lng",
    "f6",
    "f16",
    "f17",
    "f19",
    "f20",
    "f21",
    "f24",
    "f25",
    "f27",
    "f28",
    "f30",
    "D1",
    "D2",
    "D3",
    "Wind_D_Calm",
    "Wind_D_Variable",
    "Wind_D_Sin",
    "Wind_D_Cos",
    "Record_Delay_Time",
    "is_holiday",
    "has_sensitive_word",
    "Address_Freq",
    "Address_Mean"
]


missing_features = [
    column
    for column in expected_features
    if column not in df.columns
]

if missing_features:
    raise ValueError(
        f"Missing features: {missing_features}"
    )


# Keep the exact feature order expected by the model.
X_given = df[expected_features].copy()

print("Final shape:", X_given.shape)
print("Number of features:", X_given.shape[1])

X_given.head()

Final shape: (4991, 25)
Number of features: 25


,Source,Additional_Lng,f6,f16,f17,f19,f20,f21,f24,f25,...,D3,Wind_D_Calm,Wind_D_Variable,Wind_D_Sin,Wind_D_Cos,Record_Delay_Time,is_holiday,has_sensitive_word,Address_Freq,Address_Mean
0,2,1.201855,-0.687551,-0.732553,0.882512,0.534177,-0.642293,0,1,0,...,1,0,0,-1.077085,-1.057407,0.548739,1,0,0.776756,0.715034
1,1,0.735050,-0.607795,-0.159778,0.465202,0.327678,-0.642293,0,0,0,...,1,0,0,0.259554,0.394558,0.604030,0,0,-0.508124,-1.420799
2,1,-1.362486,-0.687551,0.721416,0.654888,-1.599322,-0.642293,0,0,0,...,0,1,0,0.055602,0.114939,-0.249738,0,0,0.379403,0.860464
3,2,0.076994,-0.687551,-0.424136,0.939418,-0.127208,0.325022,0,0,0,...,1,0,0,0.055602,-1.543008,0.007206,0,0,-0.343390,0.263058
4,1,-1.584539,-0.312937,1.250132,0.749732,0.363497,1.073138,0,0,0,...,1,0,0,-1.077085,1.287284,0.130799,1,0,-0.362466,-0.859038


## 04_Modeling :

In [62]:
# Load the final trained XGBoost model.

final_model = XGBClassifier()

final_model.load_model(
    "final_tuned_xgboost.json"
)


target_classes = np.load(
    "final_xgboost_classes.npy"
)

print("Final model loaded successfully.")
print("Target classes:", target_classes)

Final model loaded successfully.
Target classes: [1 2 3 4]


In [63]:
# Predict accident severity for the new data.

encoded_predictions = (
    final_model
    .predict(X_given)
    .astype(int)
)


predictions = target_classes[
    encoded_predictions
]


print(
    "Number of predictions:",
    len(predictions)
)

print(
    "Predicted classes:",
    np.unique(predictions)
)

Number of predictions: 4991
Predicted classes: [1 2 3 4]


In [66]:
encoded_predictions = (
    final_model
    .predict(X_given)
    .astype(int)
)

predictions = target_classes[
    encoded_predictions
]

In [67]:
# Evaluate the final model on the new labeled dataset.

if "y" not in df.columns:
    raise ValueError(
        "The new dataset does not contain the true target column 'y'. "
        "Evaluation metrics cannot be calculated without true labels."
    )


# Get the true labels corresponding to the processed rows.
y_given = (
    pd.to_numeric(
        df.loc[X_given.index, "y"],
        errors="raise"
    )
    .astype(int)
)


# Safety check
if len(y_given) != len(predictions):
    raise ValueError(
        f"Length mismatch: "
        f"{len(y_given)} true labels vs "
        f"{len(predictions)} predictions."
    )


# Calculate evaluation metrics.
macro_f1 = f1_score(
    y_given,
    predictions,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_given,
    predictions,
    average="weighted",
    zero_division=0
)

balanced_acc = balanced_accuracy_score(
    y_given,
    predictions
)

accuracy = accuracy_score(
    y_given,
    predictions
)


# Display results.
print("New Data Results")
print("-" * 35)

print(
    f"Macro F1:          {macro_f1:.4f}"
)

print(
    f"Weighted F1:       {weighted_f1:.4f}"
)

print(
    f"Balanced Accuracy: {balanced_acc:.4f}"
)

print(
    f"Accuracy:          {accuracy:.4f}"
)


print("\nClassification Report:\n")

print(
    classification_report(
        y_given,
        predictions,
        labels=[1, 2, 3, 4],
        digits=2,
        zero_division=0
    )
)

New Data Results
-----------------------------------
Macro F1:          0.8044
Weighted F1:       0.9042
Balanced Accuracy: 0.9098
Accuracy:          0.8986

Classification Report:

              precision    recall  f1-score   support

           1       0.62      0.86      0.72        35
           2       0.98      0.89      0.93      3938
           3       0.70      0.94      0.80       887
           4       0.63      0.95      0.76       131

    accuracy                           0.90      4991
   macro avg       0.73      0.91      0.80      4991
weighted avg       0.92      0.90      0.90      4991

